# 0. Load data

In [73]:
import pandas as pd


In [74]:
train = pd.read_csv("/Users/nick/Library/CloudStorage/OneDrive-Personal/Programming projects/Team Union 2/Team-Union/Nick/churn_train_cleaned.csv")
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6954 entries, 0 to 6953
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   GuestID         6954 non-null   int64 
 1   PromoCode       6954 non-null   object
 2   Region          6954 non-null   object
 3   AllInclusive    6954 non-null   int64 
 4   PackageType     6954 non-null   object
 5   VIP             6954 non-null   int64 
 6   RoomService     6954 non-null   int64 
 7   Dining          6954 non-null   int64 
 8   Retail          6954 non-null   int64 
 9   Spa             6954 non-null   int64 
 10  Entertainment   6954 non-null   int64 
 11  LoyaltyPoints   6954 non-null   int64 
 12  SurveyScore     6954 non-null   int64 
 13  DaysSinceEmail  6954 non-null   int64 
 14  BookingChannel  6954 non-null   object
 15  AgeGroup        6954 non-null   object
 16  ReferralSource  6954 non-null   object
 17  Churned         6954 non-null   int64 
 18  SharedRo

In [75]:
#Create X and y variables for modeling

from sklearn.model_selection import train_test_split

X = train.drop('Churned', axis=1)
y = train['Churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. XGBoost

In [76]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import shap

# Class imbalance: scale_pos_weight ≈ neg / pos in your training data
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    scale_pos_weight=neg / pos,
    eval_metric="auc",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

probs = xgb.predict_proba(X_test)[:, 1]
print(f"Test ROC-AUC:        {roc_auc_score(y_test, probs):.3f}")
print(f"Best iteration:      {xgb.best_iteration}")

# Explain individual predictions with SHAP — critical for stakeholder trust
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test, plot_type="bar")

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:PromoCode: object, Region: object, PackageType: object, BookingChannel: object, AgeGroup: object, ReferralSource: object, RoomFloor: object, RoomNumber: object, RoomSide: object

# 2. CatBoost

In [77]:
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_validate

# CatBoost handles categoricals natively — just tell it which columns are categorical.
# GuestID is an identifier, drop it from features.
drop_cols = ["GuestID", "Churned"]
X = train.drop(columns=drop_cols)
y = train["Churned"]

cat_features = [c for c in X.columns if X[c].dtype == "object"]
# CatBoost requires categorical columns to have no NaN — fill with a sentinel string.
X[cat_features] = X[cat_features].fillna("missing")

from sklearn.model_selection import cross_validate, train_test_split
X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

neg, pos = (y_train_cb == 0).sum(), (y_train_cb == 1).sum()

cat = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    scale_pos_weight=neg / pos,
    eval_metric="AUC",
    early_stopping_rounds=50,
    random_seed=42,
    verbose=False,
)

train_pool = Pool(X_train_cb, y_train_cb, cat_features=cat_features)
test_pool = Pool(X_test_cb, y_test_cb, cat_features=cat_features)

cat.fit(train_pool, eval_set=test_pool)

probs = cat.predict(X_test_cb)
print(f"Test ROC-AUC:        {roc_auc_score(y_test_cb, probs):.3f}")
print(f"Best iteration:      {cat.get_best_iteration()}")

#use scitkit-learn classification report to get precision, recall, f1-score, and support for the test set predictions
from sklearn.metrics import classification_report
y_pred = cat.predict(X_test_cb)
print(classification_report(y_test_cb, y_pred))

# Feature importance
import pandas as pd
fi = pd.DataFrame({
    "feature": X_train_cb.columns,
    "importance": cat.get_feature_importance(train_pool),
}).sort_values("importance", ascending=False)
print(fi.to_string(index=False))

Test ROC-AUC:        0.828
Best iteration:      289
              precision    recall  f1-score   support

           0       0.81      0.86      0.83       690
           1       0.85      0.80      0.82       701

    accuracy                           0.83      1391
   macro avg       0.83      0.83      0.83      1391
weighted avg       0.83      0.83      0.83      1391

       feature  importance
           Spa   12.637744
  AllInclusive   12.487487
        Region   11.543838
 Entertainment   10.878120
   RoomService    7.845341
ReferralSource    7.801650
     PromoCode    6.893249
        Dining    6.148948
     RoomFloor    5.345884
        Retail    3.938683
      RoomSide    3.237706
   PackageType    2.181411
 LoyaltyPoints    1.983649
      AgeGroup    1.939793
    RoomNumber    1.622969
DaysSinceEmail    1.549916
BookingChannel    0.916155
   SurveyScore    0.879355
           VIP    0.111761
    SharedRoom    0.056341
